### Importing necessary Libaries

In [4]:
#Standard Libaries
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import time
import warnings
warnings.filterwarnings("ignore")

print("Imports done")

import dask
print(f"Dask version: {dask.__version__}")

Imports done
Dask version: 2025.1.0


In [5]:
#Verify cluster is reachable
import subprocess

result = subprocess.run(
    ["kubectl", "get", "nodes"],
    capture_output=True, text=True
)
print(result.stdout)

NAME       STATUS   ROLES           AGE   VERSION
minikube   Ready    control-plane   17d   v1.35.1



In [6]:
#Check pods are running
result = subprocess.run(
    ["kubectl", "get", "pods"],
    capture_output=True, text=True
)
print(result.stdout)

NAME                                READY   STATUS    RESTARTS   AGE
my-dask-scheduler-bdd8df6b6-5ns58   1/1     Running   0          2m2s
my-dask-worker-5fc74c95ff-494rf     1/1     Running   0          2m2s
my-dask-worker-5fc74c95ff-q27pz     1/1     Running   0          2m2s



### LOCAL BASELINE — Dask LocalCluster

In [7]:

# Using only 1 parquet file to avoid OOM
DATA_FILES = [
    "data/0.parquet",
    "data/1.parquet",
    "data/2.parquet",
    "data/3.parquet",
    "data/4.parquet",
]

def run_reddit_pipeline(client, ddf):
    start = time.time()
    print(f"Connected to : {client}")
    print(f"Workers      : {len(client.scheduler_info()['workers'])}")
    print(f"Rows         : {len(ddf):,}")

    # Top 10 subreddits by comment count 
    print("\nRunning Analysis 1: Top subreddits by comment count...")
    top_subreddits = (
        ddf
        .query("score > 0")
        .groupby("subreddit")
        .size()
        .nlargest(10)
        .compute()
    )
    print("\n--- Top 10 Subreddits by Comment Count ---")
    print(top_subreddits.to_string())

    # Top 10 subreddits by average score 
    print("\nRunning Analysis 2: Average score per subreddit...")
    avg_score = (
        ddf
        .query("score > 0")
        .groupby("subreddit")["score"]
        .mean()
        .nlargest(10)
        .compute()
    )
    print("\n--- Top 10 Subreddits by Average Score ---")
    print(avg_score.to_string())

    # Comment volume by hour of day 
    print("\nRunning Analysis 3: Comment volume by hour...")
    ddf2 = ddf.assign(
        hour=dd.to_datetime(ddf["created_utc"], unit="s").dt.hour
    )
    hourly = (
        ddf2
        .groupby("hour")
        .size()
        .compute()
        .sort_index()
    )
    print("\n--- Comment Volume by Hour of Day ---")
    for hour, cnt in hourly.items():
        bar = "#" * (cnt // 50_000)
        print(f"  {hour:2d}:00  {cnt:>10,}  {bar}")

    elapsed = time.time() - start
    print(f"\nPipeline completed in {elapsed:.1f}s")
    return elapsed


# Load only 1 file for local baseline to avoid OOM
print("Loading 1 parquet file for local baseline...")
ddf_local = dd.read_parquet(
    "data/0.parquet",
    columns=["subreddit", "score", "created_utc"]
)

# Run on local cluster with higher memory limits
with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit="4GB") as cluster:
    with Client(cluster) as client:
        print(f"Dashboard: {client.dashboard_link}\n")
        local_time = run_reddit_pipeline(client, ddf_local)

print(f"\n⏱️ Local baseline: {local_time:.1f}s")

Loading 1 parquet file for local baseline...
Dashboard: http://127.0.0.1:8787/status

Connected to : <Client: 'tcp://127.0.0.1:60604' processes=2 threads=4, memory=7.45 GiB>
Workers      : 2
Rows         : 15,060,640

Running Analysis 1: Top subreddits by comment count...

--- Top 10 Subreddits by Comment Count ---
subreddit
AskReddit              1763728
funny                   721418
pics                    603954
gaming                  424967
atheism                 422042
politics                374826
WTF                     354394
trees                   300425
fffffffuuuuuuuuuuuu     300029
IAmA                    281981

Running Analysis 2: Average score per subreddit...

--- Top 10 Subreddits by Average Score ---
subreddit
Bestof2011             36.697409
ShitRedditWins         16.909091
listofponies           16.500000
AlignmentCharts        16.000000
SubredditDramaDrama    15.002717
2012Watch              14.424603
boring                 14.000000
lulzcakes              14.

### CONNECT TO DEPLOYED DASK CLUSTER ON MINIKUBE


In [8]:
print("Connecting to deployed Dask cluster on Minikube...")
deployed_client = Client("tcp://localhost:8786")
print(deployed_client)
print(f"\nConnected!")
print(f"Dashboard : http://localhost:8787")
print(f"Workers   : {len(deployed_client.scheduler_info()['workers'])}")

Connecting to deployed Dask cluster on Minikube...
<Client: 'tcp://10.244.0.15:8786' processes=2 threads=16, memory=13.40 GiB>

Connected!
Dashboard : http://localhost:8787
Workers   : 2


In [9]:

# LOAD DATA LOCALLY AND CONVERT TO DASK
# Workers inside pods cannot read local disk,
# so we load locally and Dask serializes the partitions to the workers over the network.

print("Loading parquet files locally...")
start_load = time.time()

# Load only 1 file for deployed test too
print("Loading 1 parquet file...")
pandas_df = pd.read_parquet(
    "data/0.parquet",
    columns=["subreddit", "score", "created_utc"]
)
print(f"Loaded {len(pandas_df):,} rows")
ddf_deployed = dd.from_pandas(pandas_df, npartitions=12)
print(f"Ready — {ddf_deployed.npartitions} partitions")

Loading parquet files locally...
Loading 1 parquet file...
Loaded 15,060,640 rows
Ready — 12 partitions


### RUN PIPELINE ON DEPLOYED CLUSTER

In [10]:
deployed_time = run_reddit_pipeline(deployed_client, ddf_deployed)

Connected to : <Client: 'tcp://10.244.0.15:8786' processes=2 threads=16, memory=13.40 GiB>
Workers      : 2
Rows         : 15,060,640

Running Analysis 1: Top subreddits by comment count...

--- Top 10 Subreddits by Comment Count ---
subreddit
AskReddit              1763728
funny                   721418
pics                    603954
gaming                  424967
atheism                 422042
politics                374826
WTF                     354394
trees                   300425
fffffffuuuuuuuuuuuu     300029
IAmA                    281981

Running Analysis 2: Average score per subreddit...

--- Top 10 Subreddits by Average Score ---
subreddit
Bestof2011             36.697409
ShitRedditWins         16.909091
listofponies           16.500000
AlignmentCharts        16.000000
SubredditDramaDrama    15.002717
2012Watch              14.424603
lulzcakes              14.000000
boring                 14.000000
suicide                14.000000
bigdickproblems        13.761905

Running A

#### Performance Comparison

In [11]:
print("=" * 50)
print(" PERFORMANCE COMPARISON")
print("=" * 50)
print(f"  Local  cluster (2 workers) : {local_time:.1f}s")
print(f"  Deployed cluster (Minikube): {deployed_time:.1f}s")
print(f"  Ratio                      : {deployed_time / local_time:.2f}x")
print("=" * 50)

 PERFORMANCE COMPARISON
  Local  cluster (2 workers) : 18.1s
  Deployed cluster (Minikube): 43.9s
  Ratio                      : 2.43x


In [12]:
deployed_client.close()
print("Client connection closed.")

Client connection closed.
